In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import json
import random
import re
import sys
import time
import urllib.request

import numpy as np
import pandas as pd
from scipy.stats import binomtest

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RUN_HEAVY = False

In [2]:
RAW_DIR       = "data/raw"
CLEAN_DIR     = "data/cleaned"
CHUNK_DIR     = "data/chunks"
EMB_DIR       = "embeddings"
VECTOR_DB_DIR = "vector_db"
BM25_DIR      = os.path.join(VECTOR_DB_DIR, "bm25s_index")
EVAL_DIR      = "evaluation"
RESULT_DIR    = os.path.join(EVAL_DIR, "results")
FIG_DIR       = "figures"

RAW_FILE   = os.path.join(RAW_DIR, "pubmed_100k_raw.csv")
CLEAN_FILE = os.path.join(CLEAN_DIR, "pubmed_cleaned.csv")
CHUNK_FILE = os.path.join(CHUNK_DIR, "pubmed_chunks.csv")

TARGET_TOTAL        = 100_000
BATCH_SIZE          = 2000
MIN_CONTENT_WORDS   = 10
APPLY_LOWERCASE     = False
CHUNK_SIZE_WORDS    = 220
CHUNK_OVERLAP_WORDS = 40
MIN_CHUNK_WORDS     = 20

MODELS = {
    "minilm":     ("sentence-transformers/all-MiniLM-L6-v2", False, 128),
    "bge":        ("BAAI/bge-base-en-v1.5", False, 64),
    "pubmedbert": ("NeuML/pubmedbert-base-embeddings", False, 64),
    "biobert":    ("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb", False, 64),
    "e5":         ("intfloat/e5-base-v2", True, 64),
}

ACTIVE_MODEL   = "bge"
RETRIEVAL_MODE = "hybrid"
TOP_K          = 8
RRF_K          = 60
CANDIDATE_POOL = 30

BASE_MODEL = "qwen2.5:7b"
FT_MODEL   = "qwen2.5-medqa-lora"
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
OLLAMA_KEEP_ALIVE = "5m"

OLLAMA_OPTIONS = {
    "num_ctx": 2048,
    "num_predict": 500,
    "num_gpu": 20,
}

GREEDY = {
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "seed": 42,
    "repeat_penalty": 1.0,
}
OLLAMA_OPTIONS.update(GREEDY)

NUM_PREDICT       = 150
JUDGE_NUM_PREDICT = 5
CHECKPOINT_EVERY  = 20
SEED              = 42

DATASET_FILES = {
    "medqa":   os.path.join(EVAL_DIR, "medqa_test.json"),
    "medmcqa": os.path.join(EVAL_DIR, "medmcqa_val.json"),
}
ANSWER_KEY = "answer_idx"

for d in (RAW_DIR, CLEAN_DIR, CHUNK_DIR, EMB_DIR, VECTOR_DB_DIR, EVAL_DIR,
          RESULT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print("project root:", os.getcwd())

project root: /home/claude/nbrun


In [3]:
COLLECT_CHECKPOINT = os.path.join(RAW_DIR, "collection_checkpoint.json")
COLLECT_FIELDS = ["id", "title", "content", "PMID"]


def load_collect_checkpoint():
    if os.path.exists(COLLECT_CHECKPOINT):
        with open(COLLECT_CHECKPOINT, "r", encoding="utf-8") as f:
            d = json.load(f)
        return d.get("raw_consumed", 0), d.get("collected", 0)
    return 0, 0


def save_collect_checkpoint(raw_consumed, collected):
    with open(COLLECT_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump({"raw_consumed": raw_consumed, "collected": collected}, f)


def collect_pubmed(target=TARGET_TOTAL):
    import csv
    from datasets import load_dataset

    raw_consumed, collected = load_collect_checkpoint()
    if collected >= target:
        print(f"already collected {collected}/{target}")
        return

    ds = load_dataset("MedRAG/pubmed", split="train", streaming=True)
    if raw_consumed:
        ds = ds.skip(raw_consumed)

    mode = "a" if (os.path.exists(RAW_FILE) and collected > 0) else "w"

    with open(RAW_FILE, mode, newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLLECT_FIELDS)
        if mode == "w":
            w.writeheader()
        batch = 0
        try:
            for row in ds:
                raw_consumed += 1
                title = (row.get("title") or "").strip()
                content = (row.get("content") or "").strip()
                if not title or not content:
                    continue
                w.writerow({"id": row.get("id"), "title": title,
                            "content": content, "PMID": row.get("PMID")})
                collected += 1
                batch += 1
                if batch >= BATCH_SIZE:
                    f.flush()
                    save_collect_checkpoint(raw_consumed, collected)
                    print(f"{collected}/{target}", end="\r")
                    batch = 0
                if collected >= target:
                    break
        finally:
            f.flush()
            save_collect_checkpoint(raw_consumed, collected)

    print(f"collected {collected}/{target} -> {RAW_FILE}")


if RUN_HEAVY:
    collect_pubmed()

In [4]:
def normalise_whitespace(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def clean_text(text):
    text = normalise_whitespace(text)
    return text.lower() if APPLY_LOWERCASE else text


def clean_corpus():
    df = pd.read_csv(RAW_FILE)
    n0 = len(df)

    df["title"] = df["title"].fillna("")
    df["content"] = df["content"].fillna("")
    df = df[(df["title"].str.strip() != "") & (df["content"].str.strip() != "")]

    df["title"] = df["title"].apply(clean_text)
    df["content"] = df["content"].apply(clean_text)

    df = df[df["content"].str.split().str.len() >= MIN_CONTENT_WORDS]
    df = df.drop_duplicates(subset=["content"], keep="first")

    df.to_csv(CLEAN_FILE, index=False)
    print(f"{n0} -> {len(df)} rows  ({n0 - len(df)} removed)  -> {CLEAN_FILE}")
    return df


if RUN_HEAVY:
    clean_corpus()

In [5]:
def chunk_text(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += step
    return chunks


def build_chunks():
    df = pd.read_csv(CLEAN_FILE)
    rows = []
    chunk_id = 0

    for _, row in df.iterrows():
        title = str(row.get("title", "")).strip()
        content = str(row.get("content", "")).strip()
        for piece in chunk_text(content):
            full = f"{title}. {piece}" if title else piece
            if len(full.split()) < MIN_CHUNK_WORDS:
                continue
            rows.append({"chunk_id": chunk_id, "doc_id": row.get("id", ""),
                         "PMID": row.get("PMID", ""), "title": title,
                         "chunk": full})
            chunk_id += 1

    out = pd.DataFrame(rows)
    out.to_csv(CHUNK_FILE, index=False)
    print(f"{len(df)} docs -> {len(out)} chunks "
          f"({len(out) / max(len(df), 1):.2f} per doc)  -> {CHUNK_FILE}")
    return out


if RUN_HEAVY:
    build_chunks()

In [6]:
def embed_all_models():
    import torch
    import faiss
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    chunks = pd.read_csv(CHUNK_FILE)["chunk"].astype(str).tolist()
    print(f"{len(chunks)} chunks, device={device}")

    for key, (hf_id, needs_prefix, batch) in MODELS.items():
        emb_path = os.path.join(EMB_DIR, f"{key}_embeddings.npy")
        idx_path = os.path.join(VECTOR_DB_DIR, f"{key}_index.faiss")
        if os.path.exists(emb_path) and os.path.exists(idx_path):
            print(f"skip {key}")
            continue

        model = SentenceTransformer(hf_id, device=device)
        texts = ["passage: " + t for t in chunks] if needs_prefix else chunks

        t0 = time.time()
        emb = model.encode(texts, batch_size=batch, show_progress_bar=True,
                           normalize_embeddings=True, convert_to_numpy=True)
        emb = np.asarray(emb).astype("float32")
        print(f"{key}: encoded in {time.time() - t0:.1f}s, dim={emb.shape[1]}")

        np.save(emb_path, emb)

        index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb)
        faiss.write_index(index, idx_path)

        del model, emb, index
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()


def build_bm25():
    import bm25s

    chunks = pd.read_csv(CHUNK_FILE)["chunk"].astype(str).tolist()
    tokens = bm25s.tokenize(chunks, stopwords="en")
    idx = bm25s.BM25()
    idx.index(tokens)
    idx.save(BM25_DIR)
    print(f"BM25 index over {len(chunks)} chunks -> {BM25_DIR}")


if RUN_HEAVY:
    embed_all_models()
    build_bm25()

In [7]:
_chunks_df = None
_dense_index = None
_dense_model = None
_dense_key = None
_bm25 = None


def load_chunks():
    global _chunks_df
    if _chunks_df is None:
        _chunks_df = pd.read_csv(CHUNK_FILE)
    return _chunks_df


def load_dense(model_key):
    global _dense_index, _dense_model, _dense_key
    import torch
    import faiss
    from sentence_transformers import SentenceTransformer

    if _dense_index is None or _dense_key != model_key:
        idx_path = os.path.join(VECTOR_DB_DIR, f"{model_key}_index.faiss")
        if not os.path.exists(idx_path):
            raise FileNotFoundError(idx_path)
        hf_id = MODELS[model_key][0]
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _dense_index = faiss.read_index(idx_path)
        _dense_model = SentenceTransformer(hf_id, device=device)
        _dense_key = model_key
    return _dense_index, _dense_model


def load_bm25():
    global _bm25
    import bm25s
    if _bm25 is None:
        if not os.path.exists(BM25_DIR):
            raise FileNotFoundError(BM25_DIR)
        _bm25 = bm25s.BM25.load(BM25_DIR, mmap=True)
    return _bm25


def dense_search(query, k, model_key):
    index, model = load_dense(model_key)
    needs_prefix = MODELS[model_key][1]
    q = ("query: " + query) if needs_prefix else query
    q_emb = np.asarray(model.encode([q], normalize_embeddings=True,
                                    convert_to_numpy=True)).astype("float32")
    scores, ids = index.search(q_emb, k)
    return list(ids[0]), list(scores[0])


def sparse_search(query, k):
    import bm25s
    bm25 = load_bm25()
    corpus_size = bm25.scores["num_docs"] if hasattr(bm25, "scores") else None
    eff_k = min(k, corpus_size) if corpus_size else k
    tokens = bm25s.tokenize(query, stopwords="en", show_progress=False)
    res, scores = bm25.retrieve(tokens, k=eff_k, show_progress=False)
    return [int(i) for i in res[0]], [float(s) for s in scores[0]]


def hybrid_search(query, k, model_key):
    dense_ids, _ = dense_search(query, CANDIDATE_POOL, model_key)
    sparse_ids, _ = sparse_search(query, CANDIDATE_POOL)

    rrf = {}
    for rank, idx in enumerate(dense_ids):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)
    for rank, idx in enumerate(sparse_ids):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)

    top = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:k]
    return [i for i, _ in top], [s for _, s in top]


def retrieve_context(query, k=None, mode=None, model_key=None):
    k = k or TOP_K
    mode = mode or RETRIEVAL_MODE
    model_key = model_key or ACTIVE_MODEL
    df = load_chunks()

    if mode == "dense":
        ids, scores = dense_search(query, k, model_key)
    elif mode == "sparse":
        ids, scores = sparse_search(query, k)
    elif mode == "hybrid":
        ids, scores = hybrid_search(query, k, model_key)
    else:
        raise ValueError(mode)

    out = []
    for idx, score in zip(ids, scores):
        row = df.iloc[int(idx)]
        out.append({"chunk": row["chunk"], "title": row.get("title", ""),
                    "doc_id": row.get("doc_id", ""),
                    "chunk_id": row.get("chunk_id", ""),
                    "PMID": row.get("PMID", ""), "score": float(score)})
    return out

In [8]:
def call_ollama(prompt, num_gpu, model=None, num_predict=None):
    import requests

    options = dict(OLLAMA_OPTIONS)
    options["num_gpu"] = num_gpu
    if num_predict is not None:
        options["num_predict"] = num_predict

    payload = {"model": model or BASE_MODEL, "prompt": prompt,
               "stream": False, "keep_alive": OLLAMA_KEEP_ALIVE,
               "options": options}

    r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    r.raise_for_status()
    return r.json().get("response", "").strip()


def generate_answer(prompt, model=None, num_predict=None):
    import requests

    try:
        out = call_ollama(prompt, OLLAMA_OPTIONS["num_gpu"], model, num_predict)
        return out or "No response generated."
    except requests.exceptions.Timeout:
        return "Ollama timeout"
    except requests.exceptions.RequestException as e:
        body = ""
        if getattr(e, "response", None) is not None:
            try:
                body = e.response.text
            except Exception:
                body = ""
        txt = (str(e) + " " + body).lower().replace("-", " ")
        oom = any(w in txt for w in ("out of memory", "cudamalloc",
                                     "unable to allocate", "failed to allocate",
                                     "bad_alloc"))
        if oom:
            time.sleep(5)
            try:
                out = call_ollama(prompt, 10, model, num_predict)
                return out or "No response generated."
            except Exception as e2:
                return "Ollama Error (after retry): " + str(e2)
        return "Ollama Error: " + str(e) + " Body: " + body
    except Exception as e:
        return str(e)


def show_decoding():
    keys = ("temperature", "top_p", "top_k", "seed", "repeat_penalty",
            "num_ctx", "num_gpu")
    active = {k: OLLAMA_OPTIONS.get(k) for k in keys if k in OLLAMA_OPTIONS}
    print("decoding:", ", ".join(f"{k}={v}" for k, v in active.items()))
    bad = [k for k in GREEDY if OLLAMA_OPTIONS.get(k) != GREEDY[k]]
    if bad:
        raise SystemExit(f"greedy settings not in force: {bad}")


def preflight(model):
    reply = generate_answer("Reply with exactly: OK", model=model, num_predict=8)
    if not isinstance(reply, str) or not reply.strip():
        raise SystemExit(f"empty reply from {model}")
    if any(w in reply.lower() for w in ("error", "connection", "refused",
                                        "not found", "try pulling")):
        raise SystemExit(f"{model} returned: {reply[:200]}")
    print(f"preflight {model}: ok ({reply.strip()[:40]!r})")

In [9]:
def build_mcqa_prompt(context_text, question, options):
    options_text = "\n".join(f"{k}) {v}" for k, v in sorted(options.items()))
    return f"""

You are a medical exam assistant.


========================

Reference Material (may or may not be directly relevant):

{context_text}


========================

Question:

{question}

Options:

{options_text}


Instructions:

Use the reference material if it helps, but rely on your own medical
knowledge as the primary basis for your answer — the reference
material is supplementary, not exhaustive.

First give a SHORT explanation in ONE sentence only (max ~25 words).
Then, on a new final line, output EXACTLY in this format (replace
<letter> with A, B, C, or D), with nothing else after it:

FINAL_ANSWER: <letter>

Answer:

"""


def build_no_context_prompt(question, options):
    options_text = "\n".join(f"{k}) {v}" for k, v in sorted(options.items()))
    return f"""

You are a medical exam assistant.


========================

Question:

{question}

Options:

{options_text}


Instructions:

Rely on your own medical knowledge to answer.

First give a SHORT explanation in ONE sentence only (max ~25 words).
Then, on a new final line, output EXACTLY in this format (replace
<letter> with A, B, C, or D), with nothing else after it:

FINAL_ANSWER: <letter>

Answer:

"""


def build_judge_prompt(context_text, question):
    return f"""

You are assessing whether retrieved reference material is useful for
answering a specific medical exam question.


========================

Reference Material:

{context_text}


========================

Question:

{question}


========================

Does the reference material contain specific information that would help
answer THIS question correctly?

Say YES only if it addresses the actual clinical or biomedical content the
question turns on. Material that is merely about the same general topic,
or that is related but does not bear on what is being asked, is NOT useful.

Reply with exactly one word: YES or NO.

Answer:

"""


def extract_choice(response_text):
    if not response_text:
        return None
    m = re.search(r"FINAL[_\s-]*ANSWER\s*[:\-]?\s*\(?\s*([ABCD])\b",
                  response_text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b([ABCD])\)", response_text)
    if m:
        return m.group(1).upper()
    return None


def extract_bare_letter(response_text, max_len=15):
    if not response_text:
        return None
    s = response_text.strip()
    if len(s) > max_len:
        return None
    m = re.search(r"\b([ABCD])\b", s, re.IGNORECASE)
    return m.group(1).upper() if m else None

In [10]:
def load_questions(dataset_key):
    with open(DATASET_FILES[dataset_key], "r", encoding="utf-8") as f:
        return json.load(f)


def check_schema(questions, name=""):
    if not questions:
        raise SystemExit(f"{name} is empty")
    missing = [k for k in ("question", "options", ANSWER_KEY)
               if k not in questions[0]]
    if missing:
        raise SystemExit(f"{name} missing {missing}; "
                         f"found {sorted(questions[0].keys())}")


def download_mcqa():
    from datasets import load_dataset

    medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
    out = [{"question": r["question"], "options": r["options"],
            "answer_idx": r["answer_idx"], "source": "medqa"} for r in medqa]
    with open(DATASET_FILES["medqa"], "w", encoding="utf-8") as f:
        json.dump(out, f)
    print("medqa:", len(out))

    med = load_dataset("openlifescienceai/medmcqa", split="validation")
    rows = []
    for r in med:
        opts = {"A": r["opa"], "B": r["opb"], "C": r["opc"], "D": r["opd"]}
        rows.append({"question": r["question"], "options": opts,
                     "answer_idx": "ABCD"[int(r["cop"])], "source": "medmcqa"})
    with open(DATASET_FILES["medmcqa"], "w", encoding="utf-8") as f:
        json.dump(rows, f)
    print("medmcqa:", len(rows))


if RUN_HEAVY:
    download_mcqa()

for ds, path in DATASET_FILES.items():
    if os.path.exists(path):
        q = load_questions(ds)
        check_schema(q, ds)
        print(f"{ds}: {len(q)} items, keys={sorted(q[0].keys())}")

medqa: 1273 items, keys=['answer_idx', 'options', 'question', 'source']
medmcqa: 4183 items, keys=['answer_idx', 'options', 'question', 'source']


In [11]:
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

HF_BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
QLORA_OUT       = "qlora_output/qwen2.5-7b-medqa-lora-v2"
TRAIN_FILE      = "finetune_data/train.jsonl"
VAL_FILE        = "finetune_data/val.jsonl"

MAX_SEQ_LENGTH              = 2048
PER_DEVICE_BATCH_SIZE       = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS                  = 3
LEARNING_RATE               = 2e-4
LOGGING_STEPS               = 10
EVAL_STEPS                  = 25
SAVE_STEPS                  = 25
SAVE_TOTAL_LIMIT            = 3
EARLY_STOPPING_PATIENCE     = 3
LOAD_BEST_MODEL_AT_END      = True


def load_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def build_example(tokenizer, prompt, completion, max_seq_length):
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    completion_ids = tokenizer.encode(completion + tokenizer.eos_token,
                                      add_special_tokens=False)

    total = len(prompt_ids) + len(completion_ids)
    if total > max_seq_length:
        overflow = total - max_seq_length
        prompt_ids = prompt_ids[min(overflow, len(prompt_ids) - 1):]

    input_ids = prompt_ids + completion_ids
    labels = ([-100] * len(prompt_ids)) + list(completion_ids)
    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}


def make_collate_fn(pad_token_id):
    import torch

    def collate(batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        ids, mask, labels = [], [], []
        for b in batch:
            pad = max_len - len(b["input_ids"])
            ids.append(b["input_ids"] + [pad_token_id] * pad)
            mask.append(b["attention_mask"] + [0] * pad)
            labels.append(b["labels"] + [-100] * pad)
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.tensor(mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)}

    return collate


def qlora_train():
    import torch
    from torch.utils.data import Dataset
    from transformers import (AutoTokenizer, AutoModelForCausalLM,
                              BitsAndBytesConfig, TrainingArguments, Trainer)
    from transformers.trainer_utils import get_last_checkpoint
    from transformers.trainer_callback import EarlyStoppingCallback
    from peft import (LoraConfig, get_peft_model,
                      prepare_model_for_kbit_training)

    class MCQADataset(Dataset):
        def __init__(self, examples, tokenizer, max_seq_length):
            self.examples = examples
            self.tokenizer = tokenizer
            self.max_seq_length = max_seq_length

        def __len__(self):
            return len(self.examples)

        def __getitem__(self, i):
            item = self.examples[i]
            return build_example(self.tokenizer, item["prompt"],
                                 item["completion"], self.max_seq_length)

    tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_ds = MCQADataset(load_jsonl(TRAIN_FILE), tokenizer, MAX_SEQ_LENGTH)
    val_ds = MCQADataset(load_jsonl(VAL_FILE), tokenizer, MAX_SEQ_LENGTH)
    print(f"train={len(train_ds)}  val={len(val_ds)}")

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)

    model = AutoModelForCausalLM.from_pretrained(HF_BASE_MODEL,
                                                 quantization_config=bnb,
                                                 device_map="auto")
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=QLORA_OUT,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS, eval_strategy="steps",
        eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
        metric_for_best_model="eval_loss", greater_is_better=False,
        bf16=True, optim="paged_adamw_8bit", gradient_checkpointing=True,
        report_to=[])

    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=make_collate_fn(tokenizer.pad_token_id),
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE)])

    last = get_last_checkpoint(QLORA_OUT) if os.path.isdir(QLORA_OUT) else None
    trainer.train(resume_from_checkpoint=last)

    model.save_pretrained(QLORA_OUT)
    tokenizer.save_pretrained(QLORA_OUT)


def merge_adapter(out_dir="models/qwen2.5-7b-medqa-merged"):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    base = AutoModelForCausalLM.from_pretrained(
        HF_BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, QLORA_OUT).merge_and_unload()

    os.makedirs(out_dir, exist_ok=True)
    merged.save_pretrained(out_dir, safe_serialization=True)
    AutoTokenizer.from_pretrained(HF_BASE_MODEL).save_pretrained(out_dir)
    print("merged ->", out_dir)


if RUN_HEAVY:
    qlora_train()
    merge_adapter()

In [12]:
ARMS = [
    ("A", "LLM only (no RAG)",        BASE_MODEL, None,     None),
    ("B", "+ naive RAG (bge, dense)", BASE_MODEL, "bge",    "dense"),
    ("C", "+ optimised retrieval",    BASE_MODEL, "minilm", "hybrid"),
    ("D", "+ fine-tuned LLM",         FT_MODEL,   "minilm", "hybrid"),
]

TAG = "full"
GATE_EMBEDDING = "minilm"


def contexts_for(dataset_key, questions, embedding_model, mode):
    if embedding_model is None:
        return None

    path = os.path.join(
        RESULT_DIR, f"contexts_{TAG}_{dataset_key}_{embedding_model}_{mode}.json")

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            ctx = json.load(f)
        if len(ctx) == len(questions):
            return ctx

    ctx = []
    for i, item in enumerate(questions):
        res = retrieve_context(item["question"], k=TOP_K, mode=mode,
                               model_key=embedding_model)
        chunks = [r["chunk"] for r in res]
        ctx.append("\n\n".join(chunks) if chunks
                   else "(no reference material retrieved)")
        if (i + 1) % 500 == 0:
            with open(path, "w", encoding="utf-8") as f:
                json.dump(ctx, f)
            print(f"  retrieved {i + 1}/{len(questions)}", end="\r")

    with open(path, "w", encoding="utf-8") as f:
        json.dump(ctx, f)
    return ctx


def gate_scores(dataset_key, questions):
    path = os.path.join(RESULT_DIR, f"gate_scores_{TAG}_{dataset_key}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        if len(df) == len(questions):
            return df

    rows = []
    for i, item in enumerate(questions):
        res = retrieve_context(item["question"], k=TOP_K, mode="dense",
                               model_key=GATE_EMBEDDING)
        s = [r["score"] for r in res] or [0.0]
        rows.append({"index": i, "top1": float(max(s)),
                     "mean_topk": float(np.mean(s))})
        if (i + 1) % 500 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df


def run_pass(name, dataset_key, questions, contexts, model, judge=False,
             tag=TAG):
    path = os.path.join(RESULT_DIR, f"{tag}_{dataset_key}_{name}.csv")

    rows, start_at = [], 0
    if os.path.exists(path):
        rows = pd.read_csv(path).to_dict("records")
        start_at = len(rows)
        if start_at >= len(questions):
            print(f"  [{name}] complete ({start_at})")
            return pd.DataFrame(rows)
        print(f"  [{name}] resuming at {start_at}/{len(questions)}")

    t0 = time.time()
    for i in range(start_at, len(questions)):
        item = questions[i]
        done = i - start_at + 1
        if done % 5 == 0:
            rate = done / max(time.time() - t0, 1e-9)
            eta = (len(questions) - i - 1) / rate / 60 if rate else 0
            print(f"  [{name} {dataset_key}] {i + 1}/{len(questions)}  "
                  f"eta {eta:.0f} min   ", end="\r")

        if judge:
            prompt, npred = build_judge_prompt(contexts[i], item["question"]), JUDGE_NUM_PREDICT
        elif contexts is None:
            prompt, npred = build_no_context_prompt(item["question"], item["options"]), NUM_PREDICT
        else:
            prompt, npred = build_mcqa_prompt(contexts[i], item["question"], item["options"]), NUM_PREDICT

        resp = generate_answer(prompt, model=model, num_predict=npred)

        if judge:
            m = re.search(r"\b(YES|NO)\b", (resp or "").strip(), re.IGNORECASE)
            rows.append({"index": i,
                         "verdict": m.group(1).upper() if m else "YES",
                         "parsed": m is not None})
        else:
            pred = extract_choice(resp)
            rows.append({"index": i, "true_answer": item[ANSWER_KEY],
                         "prediction": pred if pred is not None else "",
                         "unparseable": pred is None,
                         "correct": (pred == item[ANSWER_KEY])})

        if (i + 1) % CHECKPOINT_EVERY == 0 or i == len(questions) - 1:
            pd.DataFrame(rows).to_csv(path, index=False)

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df


def run_full_study():
    show_decoding()
    preflight(BASE_MODEL)

    for dataset_key in DATASET_FILES:
        questions = load_questions(dataset_key)
        check_schema(questions, dataset_key)
        print(f"\n{dataset_key}: {len(questions)} questions")

        ctx_cache = {}
        for _, _, _, emb, mode in ARMS:
            if emb is not None and (emb, mode) not in ctx_cache:
                ctx_cache[(emb, mode)] = contexts_for(dataset_key, questions,
                                                      emb, mode)
        gate_scores(dataset_key, questions)

        for arm_id, _, model, emb, mode in ARMS:
            if arm_id == "D":
                continue
            run_pass(arm_id, dataset_key, questions,
                     ctx_cache.get((emb, mode)) if emb else None, model)

        run_pass("JUDGE", dataset_key, questions,
                 ctx_cache[("minilm", "hybrid")], BASE_MODEL, judge=True)

        for arm_id, _, model, emb, mode in ARMS:
            if arm_id == "D":
                preflight(FT_MODEL)
                run_pass(arm_id, dataset_key, questions,
                         ctx_cache[(emb, mode)], model)


if RUN_HEAVY:
    run_full_study()

In [13]:
RCT_BASE_URL = ("https://raw.githubusercontent.com/Franck-Dernoncourt/"
                "pubmed-rct/master/PubMed_20k_RCT_numbers_replaced_with_at_sign/")
RCT_SPLITS   = ["train.txt", "dev.txt", "test.txt"]
RCT_RAW_DIR  = os.path.join(RAW_DIR, "pubmed_rct")
RCT_CHUNK_FILE = os.path.join(CHUNK_DIR, "rct_chunks.csv")
RCT_VECTOR_DIR = "vector_db_rct"
RCT_BM25_DIR   = "bm25s_index_rct"
RCT_TAG        = "rct"
RCT_EMBEDDING  = "minilm"
RCT_ANSWER_LABELS = ("RESULTS", "CONCLUSIONS")


def rct_download():
    os.makedirs(RCT_RAW_DIR, exist_ok=True)
    for split in RCT_SPLITS:
        dest = os.path.join(RCT_RAW_DIR, split)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(RCT_BASE_URL + split, dest)
        print(dest, os.path.getsize(dest))


def rct_detokenise(s):
    s = re.sub(r"\s+([,.;:!?%\)\]])", r"\1", s)
    s = re.sub(r"([\(\[])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()


def rct_parse_abstracts():
    abstracts = []
    for split in RCT_SPLITS:
        path = os.path.join(RCT_RAW_DIR, split)
        current, aid = [], None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if line.startswith("###"):
                    if current:
                        abstracts.append((aid, current))
                    aid, current = line[3:].strip(), []
                elif line.strip():
                    label, _, text = line.partition("\t")
                    if text:
                        current.append((label.strip(), rct_detokenise(text)))
        if current:
            abstracts.append((aid, current))
    print(f"{len(abstracts)} abstracts, "
          f"{sum(len(s) for _, s in abstracts)} sentences")
    return abstracts


def rct_build_chunks(conclusions_only=False):
    abstracts = rct_parse_abstracts()
    rows, chunk_id = [], 0

    for aid, sentences in abstracts:
        if conclusions_only:
            sentences = [(l, t) for l, t in sentences
                         if l in RCT_ANSWER_LABELS]
            if not sentences:
                continue
        text = " ".join(t for _, t in sentences)
        title = next((t for l, t in sentences if l == "OBJECTIVE"), "")[:200]

        for piece in chunk_text(text):
            full = f"{title}. {piece}" if title else piece
            if len(full.split()) < MIN_CHUNK_WORDS:
                continue
            rows.append({"chunk_id": chunk_id, "doc_id": aid, "PMID": aid,
                         "title": title, "chunk": full})
            chunk_id += 1

    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(RCT_CHUNK_FILE), exist_ok=True)
    df.to_csv(RCT_CHUNK_FILE, index=False)
    print(f"{len(df)} chunks -> {RCT_CHUNK_FILE}")
    return df


def rct_build_indexes(df):
    import torch
    import faiss
    import bm25s
    from sentence_transformers import SentenceTransformer

    chunks = df["chunk"].astype(str).tolist()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(MODELS[RCT_EMBEDDING][0], device=device)
    emb = np.asarray(model.encode(chunks, batch_size=128, show_progress_bar=True,
                                  normalize_embeddings=True,
                                  convert_to_numpy=True)).astype("float32")

    os.makedirs(RCT_VECTOR_DIR, exist_ok=True)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, os.path.join(RCT_VECTOR_DIR,
                                          f"{RCT_EMBEDDING}_index.faiss"))

    bm = bm25s.BM25()
    bm.index(bm25s.tokenize(chunks, stopwords="en"))
    bm.save(RCT_BM25_DIR)

    del model, emb, index
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    print("RCT indexes built")


class RCTRetriever:
    def __init__(self):
        import torch
        import faiss
        import bm25s
        from sentence_transformers import SentenceTransformer

        self.df = pd.read_csv(RCT_CHUNK_FILE)
        self.index = faiss.read_index(
            os.path.join(RCT_VECTOR_DIR, f"{RCT_EMBEDDING}_index.faiss"))
        self.bm25 = bm25s.BM25.load(RCT_BM25_DIR, mmap=True)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(MODELS[RCT_EMBEDDING][0], device=device)

    def retrieve(self, query, k=TOP_K):
        import bm25s

        q = np.asarray(self.model.encode([query], normalize_embeddings=True,
                                         convert_to_numpy=True)).astype("float32")
        _, dense_ids = self.index.search(q, CANDIDATE_POOL)
        dense_ids = list(dense_ids[0])

        res, _ = self.bm25.retrieve(
            bm25s.tokenize(query, stopwords="en", show_progress=False),
            k=min(CANDIDATE_POOL, len(self.df)), show_progress=False)
        sparse_ids = [int(i) for i in res[0]]

        rrf = {}
        for rank, i in enumerate(dense_ids):
            rrf[i] = rrf.get(i, 0.0) + 1.0 / (RRF_K + rank + 1)
        for rank, i in enumerate(sparse_ids):
            rrf[i] = rrf.get(i, 0.0) + 1.0 / (RRF_K + rank + 1)

        top = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:k]
        return [str(self.df.iloc[int(i)]["chunk"]) for i, _ in top]


def rct_evaluate(with_finetuned=False):
    show_decoding()
    retr = RCTRetriever()

    arms = [("C", BASE_MODEL)]
    if with_finetuned:
        arms.append(("D", FT_MODEL))

    for arm, model in arms:
        preflight(model)
        for dataset_key in DATASET_FILES:
            questions = load_questions(dataset_key)
            check_schema(questions, dataset_key)
            path = os.path.join(RESULT_DIR, f"{RCT_TAG}_{dataset_key}_{arm}.csv")

            done = {}
            if os.path.exists(path):
                prev = pd.read_csv(path)
                done = {int(r["index"]): r for r in prev.to_dict("records")}
                if len(done) == len(questions):
                    print(f"  {arm}/{dataset_key}: complete")
                    continue

            rows = [done[i] for i in sorted(done)]
            for i, item in enumerate(questions):
                if i in done:
                    continue
                chunks = retr.retrieve(item["question"], k=TOP_K)
                ctx = "\n\n".join(chunks) if chunks else "(no reference material retrieved)"
                resp = generate_answer(build_mcqa_prompt(ctx, item["question"],
                                                         item["options"]),
                                       model=model, num_predict=NUM_PREDICT)
                pred = extract_choice(resp)
                rows.append({"index": i, "true_answer": item[ANSWER_KEY],
                             "prediction": pred, "unparseable": pred is None,
                             "correct": (pred == item[ANSWER_KEY])})
                if (i + 1) % CHECKPOINT_EVERY == 0:
                    pd.DataFrame(rows).to_csv(path, index=False)

            df = pd.DataFrame(rows).sort_values("index")
            df.to_csv(path, index=False)
            print(f"  {arm}/{dataset_key}: {100 * df.correct.mean():.2f}%")


if RUN_HEAVY:
    rct_download()
    rct_build_indexes(rct_build_chunks())
    rct_evaluate(with_finetuned=False)

In [14]:
def load_outcomes(name):
    path = os.path.join(RESULT_DIR, name)
    if not os.path.exists(path):
        return None
    return pd.read_csv(path).set_index("index")["correct"].astype(bool)


def mcnemar(a, b):
    j = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    a2, b2 = j.a.astype(bool), j.b.astype(bool)
    gained = int(((~a2) & b2).sum())
    lost = int((a2 & (~b2)).sum())
    delta = 100.0 * (b2.mean() - a2.mean())
    p = 1.0 if gained + lost == 0 else binomtest(gained, gained + lost, 0.5).pvalue
    return round(delta, 2), gained, lost, float(p), len(j)


def gate_numbers(no_ctx, ctx):
    j = pd.concat([no_ctx.rename("a"), ctx.rename("c")], axis=1).dropna()
    a_ok, c_ok = j.a.astype(bool), j.c.astype(bool)
    n = len(j)

    a_only = int((a_ok & (~c_ok)).sum())
    c_only = int(((~a_ok) & c_ok).sum())
    d = a_only + c_only
    both = int((a_ok & c_ok).sum())

    acc_no = 100 * a_ok.mean()
    acc_ctx = 100 * c_ok.mean()
    acc_oracle = 100 * (both + d) / n

    return {"n": n,
            "acc_no_ctx": round(acc_no, 2),
            "acc_ctx": round(acc_ctx, 2),
            "acc_oracle": round(acc_oracle, 2),
            "headroom_pp": round(acc_oracle - max(acc_no, acc_ctx), 2),
            "discordant": d, "noctx_only": a_only, "ctx_only": c_only,
            "required_skill_pct": round(100 * max(a_only, c_only) / d, 2) if d else None}


def fmt_p(p):
    if p is None:
        return "-"
    return f"{p:.3g}" if p >= 1e-4 else f"{p:.1e}"


DATASETS = ("medqa", "medmcqa")

In [15]:
ladder = []
for ds in DATASETS:
    row = {"dataset": ds}
    for label, fn in (("A no retrieval",   f"full_{ds}_A.csv"),
                      ("B bge dense",      f"full_{ds}_B.csv"),
                      ("C 100k PubMed",    f"full_{ds}_C.csv"),
                      ("C 20k RCT",        f"rct_{ds}_C.csv"),
                      ("D 100k PubMed",    f"full_{ds}_D.csv"),
                      ("D 20k RCT",        f"rct_{ds}_D.csv")):
        s = load_outcomes(fn)
        row[label] = round(100 * s.mean(), 2) if s is not None else None
    ladder.append(row)

ladder_df = pd.DataFrame(ladder)
ladder_df

,dataset,A no retrieval,B bge dense,C 100k PubMed,C 20k RCT,D 100k PubMed,D 20k RCT
0,medqa,60.64,57.58,57.89,57.27,59.70,None
1,medmcqa,54.86,50.61,49.15,50.56,59.07,None


In [16]:
rows = []
for ds in DATASETS:
    A = load_outcomes(f"full_{ds}_A.csv")
    B = load_outcomes(f"full_{ds}_B.csv")
    for corpus, ctag, dtag in (("100k PubMed", f"full_{ds}_C.csv", f"full_{ds}_D.csv"),
                               ("20k RCT",     f"rct_{ds}_C.csv",  f"rct_{ds}_D.csv")):
        C, D = load_outcomes(ctag), load_outcomes(dtag)
        steps = [("A->C", A, C), ("C->D", C, D)]
        if corpus == "100k PubMed":
            steps = [("A->B", A, B), ("B->C", B, C)] + steps
        for step, x, y in steps:
            if x is None or y is None:
                continue
            delta, g, l, p, n = mcnemar(x, y)
            rows.append({"dataset": ds, "corpus": corpus, "step": step, "n": n,
                         "delta_pp": delta, "gained": g, "lost": l,
                         "p": fmt_p(p), "sig": p < 0.05})

paired_df = pd.DataFrame(rows)
paired_df

,dataset,corpus,step,n,delta_pp,gained,lost,p,sig
0,medqa,100k PubMed,A->B,1273,-3.06,114,153,0.0199,True
1,medqa,100k PubMed,B->C,1273,0.31,125,121,0.848,False
2,medqa,100k PubMed,A->C,1273,-2.75,111,146,0.0337,True
3,medqa,100k PubMed,C->D,1273,1.81,166,143,0.211,False
4,medqa,20k RCT,A->C,1273,-3.38,97,140,0.00625,True
5,medmcqa,100k PubMed,A->B,4183,-4.26,407,585,1.8e-08,True
6,medmcqa,100k PubMed,B->C,4183,-1.46,364,425,0.0326,True
7,medmcqa,100k PubMed,A->C,4183,-5.71,373,612,2.5e-14,True
8,medmcqa,100k PubMed,C->D,4183,9.92,799,384,4.8e-34,True
9,medmcqa,20k RCT,A->C,4183,-4.30,377,557,4.2e-09,True


In [17]:
head, tg, tl = [], 0, 0
for ds in DATASETS:
    for arm in ("C", "D"):
        x, y = load_outcomes(f"full_{ds}_{arm}.csv"), load_outcomes(f"rct_{ds}_{arm}.csv")
        if x is None or y is None:
            continue
        delta, g, l, p, n = mcnemar(x, y)
        tg += g
        tl += l
        head.append({"dataset": ds, "arm": arm, "n": n, "delta_pp": delta,
                     "changed_answer": g + l,
                     "changed_pct": round(100 * (g + l) / n, 1),
                     "p": fmt_p(p),
                     "verdict": "DIFFERENT" if p < 0.05 else "indistinguishable"})

corpus_df = pd.DataFrame(head)
if len(corpus_df):
    tot = int(corpus_df["n"].sum())
    pooled_p = binomtest(tg, tg + tl, 0.5).pvalue if tg + tl else 1.0
    print(f"pooled +{tg}/-{tl}  p={fmt_p(pooled_p)}   "
          f"changed {tg + tl} of {tot} ({100 * (tg + tl) / tot:.1f}%)")
corpus_df

pooled +634/-583  p=0.152   changed 1217 of 5456 (22.3%)


,dataset,arm,n,delta_pp,changed_answer,changed_pct,p,verdict
0,medqa,C,1273,-0.63,250,19.6,0.658,indistinguishable
1,medmcqa,C,4183,1.41,967,23.1,0.0621,indistinguishable


In [18]:
gates = []
for ds in DATASETS:
    A = load_outcomes(f"full_{ds}_A.csv")
    for corpus, tmpl in (("100k PubMed", f"full_{ds}_{{}}.csv"),
                         ("20k RCT",     f"rct_{ds}_{{}}.csv")):
        for arm in ("C", "D"):
            ctx = load_outcomes(tmpl.format(arm))
            if ctx is None:
                continue
            g = gate_numbers(A, ctx)
            g.update({"dataset": ds, "corpus": corpus, "arm": arm})
            gates.append(g)

gate_df = pd.DataFrame(gates)[["dataset", "corpus", "arm", "n", "acc_no_ctx",
                               "acc_ctx", "acc_oracle", "headroom_pp",
                               "discordant", "noctx_only", "ctx_only",
                               "required_skill_pct"]]
gate_df

,dataset,corpus,arm,n,acc_no_ctx,acc_ctx,acc_oracle,headroom_pp,discordant,noctx_only,ctx_only,required_skill_pct
0,medqa,100k PubMed,C,1273,60.64,57.89,69.36,8.72,257,146,111,56.81
1,medqa,100k PubMed,D,1273,60.64,59.70,72.11,11.47,304,158,146,51.97
2,medqa,20k RCT,C,1273,60.64,57.27,68.26,7.62,237,140,97,59.07
3,medmcqa,100k PubMed,C,4183,54.86,49.15,63.78,8.92,985,612,373,62.13
4,medmcqa,100k PubMed,D,4183,54.86,59.07,69.14,10.06,1018,421,597,58.64
5,medmcqa,20k RCT,C,4183,54.86,50.56,63.88,9.01,934,557,377,59.64


In [19]:
def gate_delta(a, c, d, n, s):
    return (100.0 / n) * (s * d - max(a, c))


check = []
judge_path = os.path.join(RESULT_DIR, "full_gating.csv")
if os.path.exists(judge_path):
    judge = pd.read_csv(judge_path)
    for _, r in judge.iterrows():
        sub = gate_df[(gate_df.dataset == r["dataset"]) &
                      (gate_df.corpus == "100k PubMed") &
                      (gate_df.arm == r["context_arm"])]
        if not len(sub):
            continue
        row = sub.iloc[0]
        a = int(row["noctx_only"])
        cc = int(row["ctx_only"])
        d = int(row["discordant"])
        n = int(row["n"])
        s = r["judge_correct"] / r["discordant"]
        predicted = gate_delta(a, cc, d, n, s)
        measured = 100 * (r["judge_gate_acc"] - max(r["acc_no_context"],
                                                    r["acc_always_context"]))
        check.append({"dataset": r["dataset"], "arm": r["context_arm"],
                      "judge_skill_pct": round(100 * s, 2),
                      "s_star_pct": row["required_skill_pct"],
                      "predicted_delta_pp": round(predicted, 3),
                      "measured_delta_pp": round(measured, 3),
                      "worth_deploying": bool(100 * s > row["required_skill_pct"])})

sstar_df = pd.DataFrame(check)
sstar_df

,dataset,arm,judge_skill_pct,s_star_pct,predicted_delta_pp,measured_delta_pp,worth_deploying
0,medqa,C,56.81,56.81,0.000,0.00,False
1,medqa,D,57.89,51.97,1.414,1.42,True
2,medmcqa,C,54.01,62.13,-1.913,-1.91,False
3,medmcqa,D,49.61,58.64,-2.199,-2.20,False


In [20]:
def gold_identity_check():
    out = []
    for ds in DATASETS:
        p1 = os.path.join(RESULT_DIR, f"full_{ds}_C.csv")
        p2 = os.path.join(RESULT_DIR, f"rct_{ds}_C.csv")
        if not (os.path.exists(p1) and os.path.exists(p2)):
            continue
        a = pd.read_csv(p1).set_index("index")["true_answer"]
        b = pd.read_csv(p2).set_index("index")["true_answer"]
        j = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
        out.append({"dataset": ds, "n": len(j),
                    "gold_identical": bool((j.a == j.b).all()),
                    "gold_mismatches": int((j.a != j.b).sum())})
    return pd.DataFrame(out)


def churn_breakdown():
    out = []
    for ds in DATASETS:
        p1 = os.path.join(RESULT_DIR, f"full_{ds}_C.csv")
        p2 = os.path.join(RESULT_DIR, f"rct_{ds}_C.csv")
        if not (os.path.exists(p1) and os.path.exists(p2)):
            continue
        a = pd.read_csv(p1).set_index("index")
        b = pd.read_csv(p2).set_index("index")
        j = a.join(b, lsuffix="_100k", rsuffix="_rct").dropna(
            subset=["prediction_100k", "prediction_rct"])
        changed = j["prediction_100k"] != j["prediction_rct"]
        fixed = int((changed & ~j["correct_100k"] & j["correct_rct"]).sum())
        broke = int((changed & j["correct_100k"] & ~j["correct_rct"]).sum())
        wrong_both = int(changed.sum()) - fixed - broke
        out.append({"dataset": ds, "n": len(j), "prediction_changed": int(changed.sum()),
                    "changed_pct": round(100 * changed.mean(), 1),
                    "wrong_to_right": fixed, "right_to_wrong": broke,
                    "wrong_to_wrong": wrong_both, "net": fixed - broke})
    return pd.DataFrame(out)


print(gold_identity_check().to_string(index=False))
print()
print(churn_breakdown().to_string(index=False))

dataset    n  gold_identical  gold_mismatches
  medqa 1273            True                0
medmcqa 4183            True                0

dataset    n  prediction_changed  changed_pct  wrong_to_right  right_to_wrong  wrong_to_wrong  net
  medqa 1271                 346         27.2             121             129              96   -8
medmcqa 3981                1331         33.4             474             428             429   46


In [21]:
bench_files = {
    "embedding_mcqa":     "mcqa_embedding_benchmark.csv",
    "retrieval_strategy": "retrieval_strategy_benchmark.csv",
    "llm_benchmark":      "llm_benchmark.csv",
}

for name, fn in bench_files.items():
    path = os.path.join(RESULT_DIR, fn)
    if os.path.exists(path):
        print(f"----- {name} -----")
        print(pd.read_csv(path).to_string(index=False))
        print()

----- embedding_mcqa -----
     model dataset  num_questions  accuracy  precision  recall     f1  avg_latency_sec  fallback_rate
    minilm   medqa            300    0.6200     0.6210  0.6200 0.6197             3.03         0.0000
    minilm medmcqa            300    0.4733     0.4784  0.4733 0.4746             2.56         0.0600
       bge   medqa            300    0.5967     0.5949  0.5967 0.5932             2.69         0.0000
       bge medmcqa            300    0.4500     0.4600  0.4500 0.4501             2.59         0.0533
pubmedbert   medqa            300    0.5667     0.5673  0.5667 0.5660             2.75         0.0000
pubmedbert medmcqa            300    0.4867     0.4911  0.4867 0.4863             2.59         0.0367
   biobert   medqa            300    0.5800     0.5800  0.5800 0.5800             3.03         0.0000
   biobert medmcqa            300    0.4733     0.4764  0.4733 0.4721             2.53         0.0400
        e5   medqa            300    0.5900     0.5917 

In [22]:
EXCLUDED_MODELS = ["phi3.5", "cniongolo/biomistral", "meditron:7b"]

llm = pd.read_csv(os.path.join(RESULT_DIR, "llm_benchmark.csv"))

worst = llm.groupby("llm")["fallback_rate"].max()
llm_summary = llm.pivot_table(index="llm", columns="dataset",
                              values=["accuracy", "fallback_rate",
                                      "avg_latency_sec"])
llm_summary[("worst_fallback", "")] = worst.reindex(llm_summary.index).values
llm_summary[("over_15pct", "")] = worst.reindex(llm_summary.index).gt(0.15).values
llm_summary[("excluded", "")] = [m in EXCLUDED_MODELS for m in llm_summary.index]

print("models evaluated:", llm["llm"].nunique())
print("excluded (parse failure > 50%):", len(EXCLUDED_MODELS),
      "->", EXCLUDED_MODELS)
print("retained:", llm["llm"].nunique() - len(EXCLUDED_MODELS))
print("fallback rate above 15%:", int(worst.gt(0.15).sum()),
      "->", sorted(worst[worst.gt(0.15)].index))
llm_summary

models evaluated:

 10
excluded (parse failure > 50%): 3 -> ['phi3.5', 'cniongolo/biomistral', 'meditron:7b']
retained: 7
fallback rate above 15%: 5 -> ['cniongolo/biomistral', 'koesn/llama3-openbiollm-8b', 'meditron:7b', 'medllama2:7b', 'phi3.5']


accuracy        avg_latency_sec       fallback_rate        worst_fallback over_15pct excluded
dataset                     medmcqa  medqa         medmcqa medqa       medmcqa  medqa                                   
llm                                                                                                                     
cniongolo/biomistral          0.322  0.286            2.90  3.36         0.954  0.934          0.954       True     True
gemma2:9b-instruct-q4_0       0.544  0.600            9.17  8.72         0.000  0.002          0.002      False    False
koesn/llama3-openbiollm-8b    0.418  0.436            2.87  3.95         0.258  0.208          0.258       True    False
llama3.1:8b                   0.536  0.628            4.15  5.28         0.000  0.000          0.000      False    False
llama3.2:3b                   0.458  0.512            1.55  1.77         0.002  0.000          0.002      False    False
meditron:7b                   0.334  0.284            9.13  8.08         0.932  0.932          0.932       True     True
medllama2:7b                  0.272  0.312            6.23  8.99         0.188  0.262          0.262       True    False
mistral:7b-instruct           0.440  0.502            4.97  6.07         0.006  0.006          0.006      False    False
phi3.5                        0.334  0.296            8.02  8.98         0.870  0.820          0.870       True     True
qwen2.5:7b                    0.498  0.566            2.53  2.67         0.000  0.000          0.000      False    False

In [23]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
})

C_BASE, C_CTX, C_FT, C_RCT = "#3b6ea5", "#c26a3d", "#4f8a5b", "#8a6bab"

In [24]:
stages = ["A no retrieval", "B bge dense", "C 100k PubMed", "D 100k PubMed"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

for ax, ds in zip(axes, DATASETS):
    row = ladder_df[ladder_df.dataset == ds].iloc[0]
    vals = [row[s] for s in stages]
    ax.bar(range(len(stages)), vals,
           color=[C_BASE, C_CTX, C_CTX, C_FT], width=0.62)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.4, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(stages)))
    ax.set_xticklabels(["A", "B", "C", "D"])
    ax.set_ylim(0, max(vals) * 1.18)
    ax.set_ylabel("accuracy (%)")
    ax.set_title(ds)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "nb_ladder.png"), bbox_inches="tight")
fig

<Figure size 1350x510 with 2 Axes>

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

for ax, ds in zip(axes, DATASETS):
    row = ladder_df[ladder_df.dataset == ds].iloc[0]
    labels, vals, colors = [], [], []
    for lab, col in (("A no retrieval", C_BASE),
                     ("C 100k PubMed", C_CTX),
                     ("C 20k RCT", C_RCT)):
        if pd.notna(row[lab]):
            labels.append(lab.replace(" ", "\n"))
            vals.append(row[lab])
            colors.append(col)
    ax.bar(range(len(vals)), vals, color=colors, width=0.55)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.4, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, max(vals) * 1.18)
    ax.set_ylabel("accuracy (%)")
    ax.set_title(ds)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "nb_corpus_compare.png"), bbox_inches="tight")
fig

<Figure size 1350x510 with 2 Axes>

In [26]:
sub = gate_df[gate_df.corpus == "100k PubMed"].copy()
sub["label"] = sub["dataset"] + " " + sub["arm"]

fig, ax = plt.subplots(figsize=(6.4, 3.4))
x = np.arange(len(sub))
w = 0.26

ax.bar(x - w, sub["acc_no_ctx"], w, label="no context", color=C_BASE)
ax.bar(x,      sub["acc_ctx"],   w, label="always context", color=C_CTX)
ax.bar(x + w,  sub["acc_oracle"], w, label="oracle gate", color=C_FT)

for i, r in enumerate(sub.itertuples()):
    ax.text(i + w, r.acc_oracle + 0.5, f"s*={r.required_skill_pct:.1f}%",
            ha="center", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(sub["label"])
ax.set_ylabel("accuracy (%)")
ax.set_ylim(0, sub["acc_oracle"].max() * 1.20)
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.12))

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "nb_gating.png"), bbox_inches="tight")
fig

<Figure size 960x510 with 1 Axes>

In [27]:
summary_dir = os.path.join(RESULT_DIR, "notebook_summary")
os.makedirs(summary_dir, exist_ok=True)

ladder_df.to_csv(os.path.join(summary_dir, "ladder.csv"), index=False)
paired_df.to_csv(os.path.join(summary_dir, "paired_tests.csv"), index=False)
corpus_df.to_csv(os.path.join(summary_dir, "corpus_head_to_head.csv"), index=False)
gate_df.to_csv(os.path.join(summary_dir, "gating.csv"), index=False)
sstar_df.to_csv(os.path.join(summary_dir, "sstar_validation.csv"), index=False)
gold_identity_check().to_csv(os.path.join(summary_dir, "gold_check.csv"), index=False)
churn_breakdown().to_csv(os.path.join(summary_dir, "churn.csv"), index=False)

print(sorted(os.listdir(summary_dir)))

['churn.csv', 'corpus_head_to_head.csv', 'gating.csv', 'gold_check.csv', 'ladder.csv', 'paired_tests.csv', 'sstar_validation.csv']
